# SN-CED on a free GPU — Stage 0 and Stage 0b

Two gates decide whether this architecture is worth spending money on:

| Stage | Question | Gate |
|---|---|---|
| **0** | Can a small compiler write reliable notes on real text it has never seen? | ≥95% fact retention on unseen entities |
| **0b** | With a competent model, do notes beat full context / RAG / summaries? | full-context baseline ≥70% first (Rule 2) |

Both fit one 16 GB T4. Everything checkpoints, so a killed session resumes.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Internet **On**.

In [ ]:
# 1. Get the code. Either attach the repo as a Kaggle Dataset named 'snced',
#    or upload snced.zip to the working directory.
import os, shutil, sys, pathlib

SRC = pathlib.Path('/kaggle/input/snced')
WORK = pathlib.Path('/kaggle/working/snced')
if SRC.exists() and not WORK.exists():
    shutil.copytree(SRC, WORK)
elif pathlib.Path('/kaggle/working/snced.zip').exists() and not WORK.exists():
    shutil.unpack_archive('/kaggle/working/snced.zip', WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK / 'src'))
print(os.getcwd(), os.listdir())

In [ ]:
# 2. Dependencies (torch and transformers are preinstalled on Kaggle)
!pip install -q bitsandbytes peft accelerate 2>&1 | tail -2
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Stage 0 — train the note compiler

Trains on bAbI facts hidden in WikiText prose with substituted entities, then evaluates on real
BABILong documents whose noise is book text and whose entities were held out of training.

On CPU this reached 82% retention; the open question is whether a fine-tuned encoder on GPU,
with more data, clears 95%. Stop and rerun the same cell to resume after a session ends.

In [ ]:
!python experiments/real/kaggle_stage0.py --stage compiler     --train-docs 8000 --max-steps 20000 --eval-every 500 --eval-limit 100     --finetune-encoder --encoder-lr 2e-5 --session-hours 8 --resume

In [ ]:
# Stage 0 result
import json, pathlib
m = pathlib.Path('results/checkpoints/kaggle_compiler_seed0.json')
print(json.dumps(json.loads(m.read_text()), indent=2)[:800] if m.exists() else 'not run yet')

## Stage 0b — the baseline gate

Run this **before** the full comparison. If a 7B model in 4-bit cannot answer BABILong from the
full context, no memory comparison underneath it means anything — that is exactly why the
0.5B and 1.5B attempts were stopped.

In [ ]:
!python experiments/real/kaggle_stage0.py --stage qa --gate-only     --model Qwen/Qwen2.5-7B-Instruct --bits 4 --tasks qa1 qa2 qa9 --limit 50     --session-hours 8 --resume

## Stage 0b — the five-arm comparison

Only if the gate passed. Compares full context, RAG top-k, an LLM summary, prompted notes, and
the trained compiler's notes, all answered by the same model at the same token budget.

In [ ]:
!python experiments/real/kaggle_stage0.py --stage qa     --model Qwen/Qwen2.5-7B-Instruct --bits 4 --tasks qa1 qa2 qa9 --limit 100     --budget 256 --session-hours 8 --resume

In [ ]:
# Results table
import json, pathlib, collections
rows = [json.loads(p.read_text()) for p in pathlib.Path('results/raw/kaggle_qa').glob('*.json')]
by = collections.defaultdict(dict)
for r in rows:
    by[r['extra']['task']][r['condition']] = r['accuracy']
for task, conds in sorted(by.items()):
    print(task, {c: round(a, 3) for c, a in sorted(conds.items())})

## Save the outputs

Kaggle keeps `/kaggle/working`. Download `snced_results.zip` before the session closes,
or commit the notebook so the outputs persist.

In [ ]:
!cd /kaggle/working/snced && zip -qr /kaggle/working/snced_results.zip results/raw results/logs results/checkpoints/*.json
print('saved /kaggle/working/snced_results.zip')